# 05. Interpretability and error analysis of the final model E066

**Project:** Kaggle House Prices. **Stage:** after ensembles (notebook 04).

This notebook only **displays** the tables and figures written by `experiments/interpretability/run_interpretability.py --experiment E066` (folders `outputs/interpretability/` and `outputs/figures/`). It refits nothing, does not write the registry and does not touch the holdout. The written discussion is in `docs/drafts/INTERPRETABILITY_E066.md`.

Residual = actual minus predicted on the log scale, so a negative residual is an over-prediction. Out-of-fold predictions use the seed-42 selection folds.

In [ ]:
# Resolve the project root the same way notebook 01 does: walk up until data_description.txt is found.
import os
from pathlib import Path

root = Path.cwd().resolve()
while not (root / "data_description.txt").exists():
    if root == root.parent:
        raise FileNotFoundError("Project root (data_description.txt) not found")
    root = root.parent
os.chdir(root)
print("Working directory:", Path.cwd())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.width", 170)
TAB = Path("outputs/interpretability")
FIG = Path("outputs/figures")
EID = "E066"

def show(name, width=7):
    img = mpimg.imread(FIG / f"{EID}_{name}.png")
    h, w = img.shape[:2]
    plt.figure(figsize=(width, width * h / w))
    plt.imshow(img); plt.axis("off"); plt.show()

## 1. How good is each fold, and how concentrated is the error?

In [ ]:
oof = pd.read_csv(TAB / f"{EID}_oof_predictions.csv")
r2 = oof["residual"] ** 2
summary = pd.Series({
    "rows": len(oof),
    "pooled OOF RMSE": np.sqrt(r2.mean()),
    "share of squared error, worst 5 rows": r2.nlargest(5).sum() / r2.sum(),
    "share of squared error, worst 15 rows": r2.nlargest(15).sum() / r2.sum(),
    "share of rows with |residual| > 0.2": (oof["residual"].abs() > 0.2).mean(),
    "RMSE without GrLivArea > 4000 (4 houses)": np.sqrt((oof.loc[oof["GrLivArea"] <= 4000, "residual"] ** 2).mean()),
})
display(summary.round(4).to_frame("value"))

In [ ]:
show("pred_vs_actual", 5.5)
show("residual_vs_predicted", 7)

## 2. Where it fails: price deciles, living area, neighbourhoods, worst rows

In [ ]:
display(pd.read_csv(TAB / f"{EID}_residuals_by_price_decile.csv").round(4))
show("residual_by_decile", 7)
display(pd.read_csv(TAB / f"{EID}_residuals_by_grlivarea.csv").round(4))
display(pd.read_csv(TAB / f"{EID}_residuals_by_neighborhood_top_bottom5.csv").round(4))
display(pd.read_csv(TAB / f"{EID}_worst15_rows.csv").round(3))

## 3. What the model uses: permutation importance on validation folds

Each raw column is shuffled in the validation part of each fold; the bar is the rise in RMSE. Correlated columns (GrLivArea, TotalSF and the quality interactions) share credit, so read the ranking as marginal damage, not as a partition of importance.

In [ ]:
perm = pd.read_csv(TAB / f"{EID}_permutation_importance.csv")
display(perm.head(15).round(5))
print("columns:", len(perm), "| importance > 0.001:", int((perm["importance_mean"] > 0.001).sum()),
      "| zero or negative:", int((perm["importance_mean"] <= 0).sum()))
show("permutation_top25", 7)

## 4. What the model uses: SHAP

SHAP explains the fitted model on the development rows (in-sample): it shows how the model uses its columns, not how well it generalises. One-hot levels are summed back to their raw column in the second table.

In [ ]:
display(pd.read_csv(TAB / f"{EID}_shap_mean_abs_by_raw_column.csv").head(15).round(5))
show("shap_top25_raw_bar", 7)
show("shap_beeswarm", 8)

In [ ]:
show("shap_dependence_1", 6)
show("shap_dependence_2", 6)